# 03. 학습이 무너지지 않게 — QK-clip, 이산 target 흩기, 붕괴 감지

논문 규모 학습(수십만 step, GPU 여러 대)에서는 작은 학습에서 보이지 않던 실패가 나옵니다. 이 저장소가 겪고 막아 둔 것 세 가지를 작은 예로 봅니다. 셋 다 `TrainConfig` 옵션이고, 모델 구조나 추론은 바꾸지 않습니다.

| 실패 | 막는 장치 | 코드 |
|---|---|---|
| attention logit이 끝없이 커져 행 하나에 몰린다 | QK-clip: logit 상한이 넘으면 Wq·Wk를 줄인다 | `training/stability.py` |
| 회귀 target이 같은 값으로 반복되면 출력층이 끝없이 커진다 | 반복되는 정답을 이웃 구간으로 흩는다 | `training/losses.py` |
| 손실이 갑자기 뛰어 돌아오지 않는다 | 붕괴를 알아채고 건강한 상태로 되돌린다 | `training/stability.py` |

다루는 흐름:

1. attention logit이 커지면
2. QK-clip — 상한을 재고 줄이기
3. 이산 target 흩기
4. 붕괴 감지
5. 정리

## `minipfn` 불러오기

다른 노트북과 같은 import shim입니다. 이 노트북은 `notebooks/topics_training/` 아래에 있으므로 저장소 루트는 두 단계 위입니다. 학습은 하지 않아서 몇 초면 끝납니다.

In [1]:
try:
    import minipfn
except ImportError:
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
    import minipfn

In [2]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from minipfn import set_seed
from minipfn.models.bar_distribution import FullSupportBarDistribution, get_bucket_borders
from minipfn.training.config import TrainConfig
from minipfn.training.losses import dequantize_targets
from minipfn.training.stability import CollapseGuard, apply_qk_clip, qk_logit_bounds

set_seed(0)  # 재현성을 위해 난수 시드를 고정한다.

## 1. attention logit이 커지면

v2 모델의 sample attention은 test 행 하나가 학습 행들을 보고 어느 행에서 정보를 가져올지 정합니다. 행마다의 logit $q^\top k / \sqrt{d}$를 softmax한 것이 가중치입니다. 학습 행 6개에 대한 logit 모양은 그대로 두고 크기만 키워 봅니다.

In [3]:
shape = torch.tensor([1.0, 0.8, 0.5, 0.3, 0.0, -0.4])  # 학습 행 6개에 대한 logit의 모양
sink_rows = {}
for scale in [1, 10, 100]:
    weights = torch.softmax(scale * shape, dim=-1)
    entropy = -(weights * weights.clamp(min=1e-30).log()).sum()
    sink_rows[f"logit × {scale}"] = {
        "가장 큰 logit": (scale * shape).max().item(),
        "행 1의 가중치": weights[0].item(),
        "나머지 5행의 가중치 합": weights[1:].sum().item(),
        "실질적으로 보는 행 수 (e^entropy)": entropy.exp().item(),
    }
display(pd.DataFrame(sink_rows).T.round(3))

,가장 큰 logit,행 1의 가중치,나머지 5행의 가중치 합,실질적으로 보는 행 수 (e^entropy)
logit × 1,1.0,0.283,0.717,5.447
logit × 10,10.0,0.875,0.125,1.501
logit × 100,100.0,1.000,0.000,1.000


logit이 10배가 되면 이미 행 1이 가중치 대부분을 가져가고, 100배면 다른 행은 전혀 보지 않습니다. 이렇게 한 행에 몰리는 것을 attention sink라고 합니다. softmax가 거의 one-hot이 되면 다른 행 쪽으로는 gradient도 흐르지 않아 스스로 벗어나기 어렵습니다. 이 저장소의 논문 규모 회귀 학습에서 sample attention의 logit 상한이 정상(수백 이하)에서 수만~수십만으로 커졌고, 그 뒤 학습이 불안정해져 benchmark가 크게 나빠졌습니다.

## 2. QK-clip — 상한을 재고 줄이기

logit을 매번 직접 재려면 모든 batch의 attention을 봐야 합니다. 대신 가중치만으로 상한을 잽니다. 층 입력은 LayerNorm 뒤라 토큰의 크기가 $\sqrt{E}$($E$는 토큰 폭)로 정해져 있으므로, head 하나의 logit은

$$\frac{|q^\top k|}{\sqrt{d}} \le \sigma_{\max}(W_q^\top W_k) \cdot \frac{E}{\sqrt{d}}$$

를 넘지 못합니다($d$는 head 폭, $\sigma_{\max}$는 가장 큰 특잇값). `qk_logit_bounds`가 attention마다, head마다 이 값을 냅니다. QK-clip은 상한이 기준 $\tau$를 넘는 head의 $W_q$와 $W_k$를 각각 $\sqrt{\tau / \text{상한}}$배로 줄여, 상한을 정확히 $\tau$로 맞춥니다(Kimi K2의 MuonClip과 같은 생각). 학습 중 `qk_clip_every` step마다 합니다.

작은 모델(v2 구조, 폭 64, head 4개, 층 3개)을 만들고, 한 attention의 $W_q$를 30배로 키워 폭주한 상태를 흉내 냅니다.

In [4]:
model = TrainConfig(task="classification", emsize=64, nlayers=3, nhead=4).build_model()
# qk_clip_weights와 같은 순서의 이름: 첫 층의 feature attention은 입력이 LayerNorm 뒤가 아니라 뺀다
names = [
    f"층 {layer} {kind.split('_')[0]}"
    for layer in range(len(model.layers))
    for kind in ("feature_attention", "sample_attention")
    if layer > 0 or kind == "sample_attention"
]


def bounds_table(model):
    bounds = qk_logit_bounds(model)  # (attention 수, head 수)
    return pd.DataFrame(
        bounds.double().numpy(), index=names, columns=[f"head {h}" for h in range(4)]
    )


before = bounds_table(model)
with torch.no_grad():
    kind, qkv = model.qk_clip_weights()[1]  # 층 1 feature attention
    qkv[:64].mul_(30.0)  # 이 attention의 Wq(qkv의 앞 64행)를 30배로: 폭주를 흉내 낸다
blown = bounds_table(model)
n_clipped, largest = apply_qk_clip(model, tau=100.0)
after = bounds_table(model)

states = {"처음": before, "Wq × 30": blown, "QK-clip (τ = 100) 뒤": after}
display(pd.DataFrame({k: v.max(axis=1) for k, v in states.items()}).round(1))  # head 가운데 최댓값
display(pd.DataFrame({k: v.loc["층 1 feature"] for k, v in states.items()}).T.round(1))
print(f"줄인 head: {int(n_clipped)}개, 줄이기 전 가장 큰 상한: {largest:.1f}")

,처음,Wq × 30,QK-clip (τ = 100) 뒤
층 0 sample,9.0,9.0,9.0
층 1 feature,8.7,261.2,100.0
층 1 sample,8.9,8.9,8.9
층 2 feature,9.6,9.6,9.6
층 2 sample,8.9,8.9,8.9


,head 0,head 1,head 2,head 3
처음,7.7,8.4,8.7,8.5
Wq × 30,229.7,250.9,261.2,255.8
QK-clip (τ = 100) 뒤,100.0,100.0,100.0,100.0


줄인 head: 4개, 줄이기 전 가장 큰 상한: 261.2


처음에는 모든 head의 상한이 8~10입니다. 층 1 feature attention의 Wq를 30배로 키우면 그 attention의 네 head가 230~261이 됩니다. 1절의 표로 보면 행 하나에 가중치가 모두 몰리고도 남는 크기입니다. QK-clip은 상한이 τ = 100을 넘은 그 네 head만 골라 Wq·Wk를 줄여 정확히 100에 맞추고, 다른 attention은 건드리지 않습니다. 상한은 가중치만으로 계산하므로 batch를 볼 필요가 없습니다. 이 저장소는 논문 규모 회귀 checkpoint에서 이 상한이 실제로 잰 가장 큰 logit과 거의 같은 것을 확인했습니다(678k 대 676k).

## 3. 이산 target 흩기

회귀 출력은 구간(bucket)마다의 확률입니다([01_pfn/03](../01_pfn/03_pfn_implementation_details.ipynb) 6절). 학습 손실은 정답이 떨어진 구간의 밀도 $p / \text{폭}$의 $-\log$입니다. 그런데 prior가 만든 데이터셋 가운데는 target이 별점(1~5)처럼 몇 개 값만 반복되는 것이 있습니다. 같은 값은 늘 같은 구간에 떨어지므로, 모델은 그 구간에 확률을 몰수록 손실이 계속 줄어듭니다. 확률 $p$를 1에 가깝게 만들려면 그 구간의 logit을 다른 구간보다 끝없이 크게 해야 합니다.

In [5]:
n_buckets = 100
logit_rows = {}
for p in [0.5, 0.9, 0.99, 0.999, 0.9999]:
    # 구간 100개 가운데 하나에 p, 나머지 99개에 같은 확률을 주려면 logit 차이가 이만큼
    gap = np.log(p / ((1 - p) / (n_buckets - 1)))
    logit_rows[p] = {"logit 차이": gap, "그 구간의 -log p": -np.log(p)}
table = pd.DataFrame(logit_rows).T
table.index.name = "정답 구간의 확률 p"
display(table.round(3))

,logit 차이,그 구간의 -log p
정답 구간의 확률 p,,
0.5000,4.595,0.693
0.9000,6.792,0.105
0.9900,9.190,0.010
0.9990,11.502,0.001
0.9999,13.805,0.000


$p$가 1에 다가갈수록 손실은 0에 다가가며 계속 줄지만, 필요한 logit 차이는 끝없이 커집니다. 출력층 가중치가 계속 커지고, 그 gradient가 연속 target일 때의 수백 배가 되어 학습이 무너졌습니다(논문 규모 회귀 run). 그래서 `dequantize_targets`는 한 데이터셋 안에서 **반복되는** 검증 행 정답만 골라, 구간 번호 좌표에서 ±50 구간(`dequantize_buckets` = 100) 범위의 균일 noise를 더해 이웃 구간으로 흩습니다. 연속 target, 양 끝 꼬리 구간의 값, 결측값은 그대로 둡니다. 데이터셋 하나로 확인합니다.

In [6]:
borders = get_bucket_borders(1000, (-4.0, 4.0))  # 0.008 폭 구간 1,000개
bar = FullSupportBarDistribution(borders)
stars = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
star_z = (stars - 3.0) / 1.4  # 별점을 표준화한 값
y = torch.tensor(
    [
        # 학습 행 6개
        star_z[0], star_z[2], star_z[4], star_z[2], star_z[3], 0.37,
        # 검증 행 6개: 반복되는 별점 4개, 연속값 1개, 결측 1개
        star_z[2], star_z[4], star_z[2], star_z[1], -0.913, float("nan"),
    ]
)[None]  # (데이터셋 1, 행 12)  # fmt: skip
n_train = 6
torch.manual_seed(0)
spread = dequantize_targets(y, n_train, bar, n_buckets=100.0)

y_eval = y[0, n_train:]
spread_eval = spread[0]


def bucket_of(values):
    # 결측은 구간이 없다 ("—")
    index = bar._bucket_index(torch.nan_to_num(values).contiguous()).numpy()
    return ["—" if np.isnan(v) else int(k) for v, k in zip(values.numpy(), index, strict=True)]


display(
    pd.DataFrame(
        {
            "정답": y_eval.numpy(),
            "정답의 구간": bucket_of(y_eval),
            "흩은 정답": spread_eval.numpy(),
            "흩은 정답의 구간": bucket_of(spread_eval),
            "바뀜": ~torch.isclose(y_eval, spread_eval, equal_nan=True).numpy(),
        },
        index=["별점 3", "별점 5", "별점 3", "별점 2", "연속값", "결측"],
    ).round(3)
)

,정답,정답의 구간,흩은 정답,흩은 정답의 구간,바뀜
별점 3,0.000,500,-0.003,499,True
별점 5,1.429,678,1.643,705,True
별점 3,0.000,500,-0.329,458,True
별점 2,-0.714,410,-0.714,410,False
연속값,-0.913,385,-0.913,385,False
결측,NaN,—,NaN,—,False


검증 행의 별점 3(두 번)과 별점 5는 데이터셋 안에서 반복되는 값이라 이웃 구간으로 흩어졌습니다. 같은 별점 3이라도 하나는 499번, 하나는 458번 구간으로 가서 더는 한 구간에 몰리지 않습니다. 별점 2는 이 데이터셋에서 한 번만 나와 반복이 아니므로 연속값·결측과 함께 그대로입니다. 흩는 것은 손실을 재는 검증 행의 정답뿐이고, context로 들어가는 학습 행의 정답은 그대로 둡니다.

## 4. 붕괴 감지

논문 규모 run에서는 gradient가 튀지 않았는데도 수백 step 안에 손실이 뛰어 모델이 상수만 내는 상태로 무너진 적이 있습니다. 같은 seed로 다시 돌리면 무너지지 않았습니다(GPU 계산의 비결정성). 그래서 막기보다 알아채고 되돌립니다. `CollapseGuard`는 `log_every` step마다 그 사이 평균 손실(window 평균)을 받아, 최근 건강한 window들의 중앙값보다 `collapse_threshold`(0.5) 넘게 높은 window가 `collapse_patience`(3)번 이어지면 붕괴로 봅니다. 학습 루프는 그때 `snapshot_every` step마다 메모리에 남겨 둔 건강한 상태로 되돌리고, 데이터 seed를 바꿔 다시 갑니다.

손실 기록 두 가지를 넣어 봅니다. 하나는 한 번 튀었다가 돌아오고, 다른 하나는 뛴 채로 머뭅니다.

In [7]:
healthy = [2.0, 1.9, 1.8, 1.75, 1.7, 1.68, 1.66]
histories = {
    "한 번 튐": healthy + [2.4, 1.65, 1.64, 1.63],
    "뛴 채로 머묾": healthy + [2.4, 2.5, 2.45, 2.5],
}
guard_rows = []
for name, losses in histories.items():
    guard = CollapseGuard(threshold=0.5, patience=3)
    for window, loss in enumerate(losses):
        collapsed = guard.update(loss)
        guard_rows.append(
            {
                "기록": name,
                "window": window,
                "평균 손실": loss,
                "나쁜 window 연속": guard.bad,
                "붕괴": collapsed,
            }
        )
guard_table = pd.DataFrame(guard_rows)
later = guard_table[guard_table["window"] >= 6]  # 건강한 7개 window 뒤부터
display(later.set_index(["기록", "window"]))

평균 손실  나쁜 window 연속     붕괴
기록      window                            
한 번 튐   6        1.66             0  False
        7        2.40             1  False
        8        1.65             0  False
        9        1.64             0  False
        10       1.63             0  False
뛴 채로 머묾 6        1.66             0  False
        7        2.40             1  False
        8        2.50             2  False
        9        2.45             3   True
        10       2.50             4   True

두 기록 모두 7번째 window에서 2.40으로 뜁니다. 건강한 window들의 중앙값(1.75)보다 0.5 넘게 높으므로 나쁜 window 하나입니다. "한 번 튐"은 다음 window에서 1.65로 돌아와 연속 횟수가 0으로 돌아가고, 붕괴로 보지 않습니다. "뛴 채로 머묾"은 나쁜 window가 세 번 이어진 9번째 window에서 붕괴로 판단합니다. 학습 루프는 이때 건강했던 snapshot으로 되돌립니다.

## 5. 정리

| 장치 | 학습 설정 | 기본값 |
|---|---|---|
| QK-clip | `qk_clip`(상한 τ), `qk_clip_every` | 끔(`None`), 50 step |
| 이산 target 흩기 | `dequantize_buckets` | 100 구간 |
| 붕괴 감지와 되돌리기 | `collapse_threshold`, `collapse_patience`, `snapshot_every`, `max_rewinds`, `rewind_lr_factor` | 0.5, 3, 2,000 step, 10번, 1.0 |

- QK-clip은 가중치만으로 logit 상한을 재므로 batch를 보지 않고도 attention sink를 막습니다(1·2절).
- 이산 target 흩기는 반복되는 검증 행 정답만 바꿔, 최적 logit을 유한하게 만듭니다(3절).
- 붕괴 감지는 한 번 튀는 것은 넘기고, 뛴 채로 머무는 것만 되돌립니다(4절). 되돌린 횟수는 MLflow `rewinds`로 남습니다.